# EAGLE：从特征草稿到并行验证的投机解码

**面试问题：EAGLE 为什么预测特征而不是只预测 Token，接受率与加速怎样计算？**

## 回答主线

1. 自回归基线每生成一个 Token 都调用一次目标模型，延迟随输出长度线性增长。
2. EAGLE 用轻量草稿头外推目标模型的中间特征，再由同一个 LM Head 得到多个候选 Token。
3. 目标模型一次前向并行验证整段候选，只接受从左到右连续正确的前缀。
4. 首个错误位置必须由目标模型纠正，因此算法保持目标分布，而不是盲信草稿。
5. 真正的收益取决于平均连续接受长度、草稿开销、验证批大小和内存带宽。
6. 线上要按请求记录接受直方图，并在低接受领域缩短草稿长度。

## 真实案例

客服模型要生成退款、密码、发票、物流和取消订单五类短回复。我们用五条脱敏工单和可读 Token 序列模拟目标模型；草稿特征由受控分数矩阵表示，以便逐位置看到候选、接受和纠正。这里不训练真实 EAGLE 网络，只复现“特征候选—并行验证—前缀接受—自适应长度”的关键协议。教学实验使用可读的小数据解释机制，结果不能外推为线上收益。

### 输入预览：五条工单、目标回复与草稿特征

In [1]:
import numpy as np  # 导入数组运算以表示特征分数和统计指标。

tickets = [  # 构造五条有业务含义的脱敏客服工单。
    {"id": "T101", "query": "退款多久到", "target": ["退款", "将在", "3天", "到账", "。"], "draft": ["退款", "将在", "5天", "到账", "。"]},  # 设置政策天数容易出错的退款工单。
    {"id": "T102", "query": "忘记密码", "target": ["请", "重置", "密码", "。"], "draft": ["请", "重置", "密码", "。"]},  # 设置草稿完全正确的密码工单。
    {"id": "T103", "query": "发票在哪里", "target": ["发票", "已", "发送", "。"], "draft": ["发票", "将", "发送", "。"]},  # 设置第二个 Token 发生分歧的发票工单。
    {"id": "T104", "query": "包裹何时到", "target": ["包裹", "明天", "送达", "。"], "draft": ["包裹", "明天", "送达", "。"]},  # 设置草稿完全正确的物流工单。
    {"id": "T105", "query": "取消订单", "target": ["订单", "已", "取消", "。"], "draft": ["订单", "无法", "取消", "。"]},  # 设置状态判断错误的取消工单。
]  # 完成五条样本定义。
extra_domain_tokens = ["需要", "人工", "复核", "可以", "自动", "通过"]  # 预留后续领域漂移反例会出现的草稿与目标 Token。
vocabulary = sorted({token for item in tickets for key in ("target", "draft") for token in item[key]}.union(extra_domain_tokens))  # 汇总教学词表以模拟共享 LM Head。
token_to_id = {token: index for index, token in enumerate(vocabulary)}  # 建立 Token 到特征维度的映射。
print(f"教学词表大小={len(vocabulary)}，样本数={len(tickets)}")  # 输出实验规模。
for item in tickets:  # 逐条预览目标序列和草稿序列。
    print(f"{item['id']} 查询={item['query']:<8} 目标={' '.join(item['target'])} 草稿={' '.join(item['draft'])}")  # 让错误位置具备可读业务语义。

教学词表大小=25，样本数=5
T101 查询=退款多久到    目标=退款 将在 3天 到账 。 草稿=退款 将在 5天 到账 。
T102 查询=忘记密码     目标=请 重置 密码 。 草稿=请 重置 密码 。
T103 查询=发票在哪里    目标=发票 已 发送 。 草稿=发票 将 发送 。
T104 查询=包裹何时到    目标=包裹 明天 送达 。 草稿=包裹 明天 送达 。
T105 查询=取消订单     目标=订单 已 取消 。 草稿=订单 无法 取消 。


## Baseline 基线：目标模型逐 Token 解码

In [2]:
def autoregressive_baseline(items):  # 实现每轮只生成一个 Token 的目标模型基线。
    rows = []  # 收集每条工单的调用次数和输出。
    for item in items:  # 遍历所有客服工单。
        generated = []  # 初始化当前回复。
        calls = 0  # 统计目标模型串行调用次数。
        for token in item["target"]:  # 按目标模型的贪心序列逐 Token 生成。
            calls += 1  # 每生成一个 Token 支付一次目标前向成本。
            generated.append(token)  # 把当前 Token 提交到回复。
        rows.append({"id": item["id"], "output": generated, "target_calls": calls})  # 保存当前基线结果。
    return rows  # 返回逐请求基线账本。

baseline_rows = autoregressive_baseline(tickets)  # 运行五条工单的自回归基线。
baseline_calls = sum(row["target_calls"] for row in baseline_rows)  # 汇总目标模型调用次数。
print("工单   目标调用  基线输出")  # 输出基线表头。
for row in baseline_rows:  # 逐工单展示串行成本。
    print(f"{row['id']} {row['target_calls']:>8}  {' '.join(row['output'])}")  # 展示每个 Token 一次调用的结果。
print(f"基线总目标调用={baseline_calls}")  # 给出后续投机解码的同口径对照。

工单   目标调用  基线输出
T101        5  退款 将在 3天 到账 。
T102        4  请 重置 密码 。
T103        4  发票 已 发送 。
T104        4  包裹 明天 送达 。
T105        4  订单 已 取消 。
基线总目标调用=21


### 核心实现：特征草稿、共享 LM Head 与前缀验证

In [3]:
def token_feature(token, confidence=4.0):  # 把草稿 Token 编码为可由共享 LM Head 解码的特征向量。
    feature = np.full(len(vocabulary), -1.0, dtype=float)  # 初始化其他 Token 的低分特征。
    feature[token_to_id[token]] = confidence  # 给草稿预测 Token 设置最高特征分数。
    return feature  # 返回教学版隐藏特征。

def feature_draft(item, start, width):  # 模拟 EAGLE 草稿头一次外推多个未来特征。
    proposed_tokens = item["draft"][start:start + width]  # 读取当前位置之后的轻量草稿候选。
    features = np.stack([token_feature(token) for token in proposed_tokens]) if proposed_tokens else np.empty((0, len(vocabulary)))  # 形成候选特征矩阵。
    decoded = [vocabulary[int(np.argmax(row))] for row in features]  # 使用共享 LM Head 的 argmax 解码候选 Token。
    return decoded, features  # 返回 Token 候选和可检查的特征。

def verify_prefix(item, draft_width):  # 实现目标模型并行验证与连续前缀接受。
    position = 0  # 从回复第一个位置开始。
    output = []  # 收集最终保持目标模型一致的输出。
    trace = []  # 记录每轮候选、接受长度和纠正 Token。
    while position < len(item["target"]):  # 在目标回复完成前持续验证。
        proposal, features = feature_draft(item, position, draft_width)  # 一次生成至多 draft_width 个草稿特征。
        gold_slice = item["target"][position:position + len(proposal)]  # 目标模型一次并行得到相同位置的验证 Token。
        accepted = 0  # 初始化连续接受长度。
        for draft_token, gold_token in zip(proposal, gold_slice):  # 从左到右比较草稿和目标 Token。
            if draft_token != gold_token:  # 一旦遇到首个不一致就停止接受。
                break  # 后续候选依赖错误前缀，不能继续提交。
            accepted += 1  # 当前候选与目标一致时扩展接受前缀。
        output.extend(proposal[:accepted])  # 提交连续正确的草稿前缀。
        correction = None  # 默认本轮无需目标模型纠正。
        if accepted < len(gold_slice):  # 草稿块中存在首个错误位置。
            correction = gold_slice[accepted]  # 取目标模型在首错位置的 Token。
            output.append(correction)  # 提交目标 Token 保证输出正确性。
        consumed = accepted + (1 if correction is not None else 0)  # 计算本轮实际推进的位置数。
        trace.append({"start": position, "proposal": proposal, "accepted": accepted, "correction": correction, "feature_shape": features.shape})  # 保存完整验证账本。
        position += max(consumed, 1)  # 推进到尚未生成的位置并防止空循环。
    return output, trace  # 返回最终输出和逐轮验证轨迹。

demo_output, demo_trace = verify_prefix(tickets[0], draft_width=3)  # 对退款工单演示首错纠正。
print("退款工单验证轨迹：")  # 输出可读轨迹标题。
for round_id, event in enumerate(demo_trace, start=1):  # 逐轮展示并行验证过程。
    print(f"round={round_id} start={event['start']} proposal={event['proposal']} accepted={event['accepted']} correction={event['correction']} features={event['feature_shape']}")  # 展示连续接受和目标纠错。
print("最终输出：", demo_output)  # 验证错误的五天没有进入最终回复。

退款工单验证轨迹：
round=1 start=0 proposal=['退款', '将在', '5天'] accepted=2 correction=3天 features=(3, 25)
round=2 start=3 proposal=['到账', '。'] accepted=2 correction=None features=(2, 25)
最终输出： ['退款', '将在', '3天', '到账', '。']


## 结果解读：同一输出下比较目标调用与接受率

In [4]:
speculative_rows = []  # 收集五条工单的投机解码指标。
for item in tickets:  # 对相同数据运行 draft width 为三的验证。
    output, trace = verify_prefix(item, draft_width=3)  # 执行特征草稿与目标验证。
    proposed = sum(len(event["proposal"]) for event in trace)  # 统计草稿提出的 Token 数。
    accepted = sum(event["accepted"] for event in trace)  # 统计被目标接受的草稿 Token 数。
    speculative_rows.append({"id": item["id"], "output": output, "rounds": len(trace), "accepted": accepted, "proposed": proposed})  # 保存逐请求指标。
print("工单   验证轮数  接受/提出  接受率   输出正确")  # 输出投机解码结果表头。
for row, item in zip(speculative_rows, tickets):  # 将结果与目标序列逐项对照。
    rate = row["accepted"] / row["proposed"]  # 计算当前请求草稿接受率。
    print(f"{row['id']} {row['rounds']:>9} {row['accepted']:>3}/{row['proposed']:<3} {rate:>7.1%} {row['output'] == item['target']}")  # 展示效率与正确性。
speculative_calls = sum(row["rounds"] for row in speculative_rows)  # 汇总并行验证轮数。
call_reduction = 1.0 - speculative_calls / baseline_calls  # 计算未扣除草稿开销的目标调用降幅。
print(f"目标调用 {baseline_calls} -> {speculative_calls}，理论调用降幅={call_reduction:.1%}")  # 给出同口径核心结果。
print("解读：高接受请求一轮验证多个 Token；首错仍由目标模型纠正，所以加速不会改变本例目标输出。")  # 解释收益和正确性来源。

工单   验证轮数  接受/提出  接受率   输出正确
T101         2   4/5     80.0% True
T102         2   4/4    100.0% True
T103         2   3/5     60.0% True
T104         2   4/4    100.0% True
T105         2   3/5     60.0% True
目标调用 21 -> 10，理论调用降幅=52.4%
解读：高接受请求一轮验证多个 Token；首错仍由目标模型纠正，所以加速不会改变本例目标输出。


## 失败案例：低接受领域固定长草稿反而浪费

In [5]:
bad_ticket = {"id": "T106", "query": "高风险拒付", "target": ["需要", "人工", "复核", "。"], "draft": ["可以", "自动", "通过", "。"]}  # 构造草稿从首 Token 起就错误的领域漂移请求。
long_output, long_trace = verify_prefix(bad_ticket, draft_width=4)  # 使用固定长草稿验证低接受请求。
short_output, short_trace = verify_prefix(bad_ticket, draft_width=1)  # 降级为每轮一个候选以减少无效草稿计算。
long_proposed = sum(len(event["proposal"]) for event in long_trace)  # 统计长草稿产生的候选总数。
short_proposed = sum(len(event["proposal"]) for event in short_trace)  # 统计短草稿产生的候选总数。
print(f"固定 width=4：验证轮数={len(long_trace)}，草稿 Token={long_proposed}，输出={' '.join(long_output)}")  # 展示低接受时的无效候选。
print(f"自适应 width=1：验证轮数={len(short_trace)}，草稿 Token={short_proposed}，输出={' '.join(short_output)}")  # 展示缩短草稿后的开销变化。
print("修正策略：按最近接受长度分桶；连续首错时缩到 1，高接受稳定后再逐步放大。")  # 给出可落地的在线控制策略。

固定 width=4：验证轮数=4，草稿 Token=10，输出=需要 人工 复核 。
自适应 width=1：验证轮数=4，草稿 Token=4，输出=需要 人工 复核 。
修正策略：按最近接受长度分桶；连续首错时缩到 1，高接受稳定后再逐步放大。


### 生产边界与观测合同

In [6]:
acceptance_histogram = {row["id"]: [event["accepted"] for event in verify_prefix(item, 3)[1]] for row, item in zip(speculative_rows, tickets)}  # 构造请求级连续接受长度直方图。
serving_contract = {"draft_model": "eagle-head-r3", "target_model": "support-7b-r8", "max_draft": 3, "metric": "accepted_prefix_length", "fallback": "target-only"}  # 固化版本、上限、指标和降级策略。
print("接受长度账本：", acceptance_histogram)  # 展示定位领域退化所需的逐请求数据。
print("Serving 合同：", serving_contract)  # 展示线上必须一起版本化的配置。
print("生产替换点：真实 EAGLE 需要训练特征预测器、树形候选、采样分布校正、KV 复用和融合验证 Kernel。")  # 明确教学协议与真实系统差距。

接受长度账本： {'T101': [2, 2], 'T102': [3, 1], 'T103': [1, 2], 'T104': [3, 1], 'T105': [1, 2]}
Serving 合同： {'draft_model': 'eagle-head-r3', 'target_model': 'support-7b-r8', 'max_draft': 3, 'metric': 'accepted_prefix_length', 'fallback': 'target-only'}
生产替换点：真实 EAGLE 需要训练特征预测器、树形候选、采样分布校正、KV 复用和融合验证 Kernel。


## 回归测试：最后只保护输出一致性与接受协议

In [7]:
assert all(row["output"] == item["target"] for row, item in zip(speculative_rows, tickets))  # 验证投机解码不改变五条目标输出。
assert speculative_calls < baseline_calls  # 验证高接受样本使目标验证轮数低于逐 Token 基线。
assert demo_trace[0]["accepted"] == 2 and demo_trace[0]["correction"] == "3天"  # 验证退款政策错误在首错位置被目标纠正。
assert long_output == short_output == bad_ticket["target"]  # 验证自适应草稿宽度只改变成本不改变结果。
assert short_proposed < long_proposed  # 验证低接受领域缩短草稿减少无效候选。
print("回归测试通过：目标一致性、连续前缀、首错纠正、调用降幅和自适应草稿均成立。")  # 用少量断言总结核心合同。

回归测试通过：目标一致性、连续前缀、首错纠正、调用降幅和自适应草稿均成立。
